In [36]:
library(rtracklayer)
library(tidyverse)
library(edgeR)
library(DESeq2)

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count



Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, 

# read in cRE annotations and convert to GRanges

In [16]:
BG  <- "/mnt/disk/home/agulka/projects/PROseq_IFN/PROseq_GM12878_IFN/bedgraph"
CRE <- "/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/bed/ATAC_peaks/filtered/consensus_filt_annot.tsv"
ENSEMBL <- "/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/ENSEMBL/ensembl.canonical.prot-coding.transcript.tsv"
# read in cRE annotations
cre <- read.delim(CRE)

#convert to GRanges
gr  <- GRanges(cre$seqnames, IRanges(cre$start, cre$end))
#annotate based on whether cRE resides within protein-coding gene
ens <- read.delim(ENSEMBL)
cre$in_gene <- overlapsAny(gr, GRanges(paste0("chr", ens$Chromosome.scaffold.name),
                                       IRanges(ens$Transcript.start..bp.,
                                               ens$Transcript.end..bp.)))
head(cre)

,seqnames,start,end,width,strand,TSS,H3K4me3,H3K4me1,H3K27ac,H3K27me3,CTCF,type,inDMSO,inBRM,ID,in_gene
,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<lgl>,<lgl>,<int>,<lgl>
1,chr1,778361,779215,855,*,TRUE,TRUE,FALSE,TRUE,FALSE,TRUE,prom,TRUE,TRUE,1,FALSE
2,chr1,804416,805076,661,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,TRUE,2,FALSE
3,chr1,817194,817588,395,*,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,other_TSS,TRUE,FALSE,3,FALSE
4,chr1,826851,828112,1262,*,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,prom,TRUE,TRUE,4,FALSE
5,chr1,838326,838675,350,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,FALSE,5,FALSE
6,chr1,857920,859154,1235,*,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,other_TSS,TRUE,TRUE,6,FALSE


# generate counts matrix

In [3]:
# calculate total bedGraph signal (+ and - strand) per cRE
#counting function
count_bedgraph <- function(f) {
    bg  <- import(f, format = "bedGraph")
    ov  <- findOverlaps(gr, bg)
    v   <- tapply(bg$score[subjectHits(ov)], queryHits(ov), sum)
    out <- numeric(length(gr))
    out[as.integer(names(v))] <- v
    out
}

In [17]:
libs   <- sub("_hg38_dedup_3pr_forward[.]bedGraph$", "",
              basename(Sys.glob(file.path(BG, "*_3pr_forward.bedGraph")))) # simplify sample names

counts <- sapply(libs, function(l)
              count_bedgraph(sprintf("%s/%s_hg38_dedup_3pr_forward.bedGraph", BG, l)) +
              count_bedgraph(sprintf("%s/%s_hg38_dedup_3pr_reverse.bedGraph",  BG, l))) # for each library, count + and - strand 
colnames(counts) <- sub("^Gorkin001_PR_[0-9]+_", "", libs)

# ID is 1..111721 in file order, so this stays row-aligned with the annotation
# and with the DAR tables, which join to it on ID.
cRE_counts <- data.frame(ID = cre$ID, cre[, c("seqnames", "start", "end", "type")],
                         counts, check.names = FALSE)

# write out counts table

In [120]:

write.table(cRE_counts, "PROseq_cRE_counts.tsv", sep='\t', quote=F, row.names=F)

In [34]:
# make row and column metadata

In [297]:
counts_only <- cRE_counts[,-1:-5]
cpm <- apply(counts_only, 2, function(x) 1000000*x/sum(x))
row.data <- cRE_counts[,1:5]
col.data <- data.frame(sample=colnames(cRE_counts[,-1:-5])) %>% mutate(condition=sub("_.$", "", sample), rep=sub("^.*_", "", sample))

In [228]:
keep <- which(apply(cpm, 1, function(x) max(x)) >= 1)
counts_filt <- counts_only[keep,]
row.data_filt <- row.data[keep,]

# differential PRO-seq counts (~ eRNA) at cREs

In [263]:
dds <- DESeqDataSetFromMatrix(countData=counts_only[keep,], colData=col.data, design=~condition)
des <- DESeq(dds)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [239]:
deg_res <- function (des, row.data, contrast) 
{
    res <- results(des, contrast = contrast)
    res <- cbind(row.data, data.frame(res)) %>% 
            mutate(change = case_when(log2FoldChange > log2(1.5) & 
                                      padj < 0.05 ~ "up", 
                                      log2FoldChange < -log2(1.5) & padj < 0.05 ~ "down", 
                                      TRUE ~ "ns"))
    return(res)
}

In [264]:
bafinh <- deg_res(des, row.data[keep,], contrast=c("condition", "BAFinh_veh", "veh_veh"))
ifn <- deg_res(des, row.data[,keep], contrast=c("condition", "veh_IFN", "veh_veh"))
bafinh_ifn <- deg_res(des, row.data[,keep], contrast=c("condition", "BAFinh_IFN", "veh_IFN"))

ERROR: Error in `[.data.frame`(row.data, , keep): undefined columns selected


In [241]:
dds <- DESeqDataSetFromMatrix(countData=counts_filt[grep("poised_enh", row.data_filt$type),], colData=col.data, design=~condition)
des <- DESeq(dds)
bafinh <- deg_res(des, row.data[grep("poised_enh", row.data_filt$type),], contrast=c("condition", "BAFinh_veh", "veh_veh"))
bafinh %>% group_by(change) %>% summarize(n())

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



change,n()
<chr>,<int>
down,531
ns,6051
up,120


In [72]:
write.table(bafinh, "BAFinh_v_Veh_PROseq_cRE.tsv", sep='\t', quote=F, row.names=F)
write.table(ifn, "IFN_v_Veh_PROseq_cRE.tsv", sep='\t', quote=F, row.names=F)
write.table(bafinh_ifn, "BAFinh_IFN_v_IFN_PROseq_cRE.tsv", sep='\t', quote=F, row.names=F)

In [266]:
library(here)
ens <- read.delim(
    here("inputs","ENSEMBL","ensembl.canonical.prot-coding.transcript.tsv"), sep='\t')

deg <- read.delim(
    here("inputs","DEGs", "GM12878", "BAFinh_stimulus","PROseq", "PRO_DEGs_IFN_Veh.tsv"), sep='\t')
counts <- read.delim(
    here("inputs","DEGs","GM12878","BAFinh_stimulus","PROseq","PRO_counts_genes.tsv"), sep='\t')


GENE_CPM <- 1   # veh_IFN CPM at or above which a host gene counts as transcribed
cpm  <- apply(counts, 2, function(x) 1000000*x/sum(x))
veh_cpm <- rowMeans(cpm[, grepl("veh_veh", colnames(cpm))])
ifn_cpm <- rowMeans(cpm[, grepl("veh_IFN", colnames(cpm))])
txn_veh  <- deg$ENSEMBL_ID[veh_cpm >= GENE_CPM]
txn_ifn  <- deg$ENSEMBL_ID[ifn_cpm >= GENE_CPM]

gr_veh_txn<- GRanges(paste0("chr", ens$Chromosome.scaffold.name),
                  IRanges(ens$Transcript.start..bp., ens$Transcript.end..bp.))[
                  ens$Gene.stable.ID %in% txn_veh]


In [275]:
bafinh %>% group_by(inside=overlapsAny(gr[keep,], gr_veh_txn), type, change) %>%
    summarize(n()) %>% filter(type=="poised_enh")

bafinh %>% group_by(inside=overlapsAny(gr[keep,], gr_veh_txn), type, change) %>%
    summarize(n()) %>% filter(type=="CTCF_only")

`summarise()` has grouped output by 'inside', 'type'. You can override using
the `.groups` argument.


inside,type,change,n()
<lgl>,<chr>,<chr>,<int>
FALSE,poised_enh,down,372
FALSE,poised_enh,ns,1948
FALSE,poised_enh,up,91
TRUE,poised_enh,down,181
TRUE,poised_enh,ns,3883
TRUE,poised_enh,up,177


`summarise()` has grouped output by 'inside', 'type'. You can override using
the `.groups` argument.


inside,type,change,n()
<lgl>,<chr>,<chr>,<int>
FALSE,CTCF_only,down,66
FALSE,CTCF_only,ns,1527
FALSE,CTCF_only,up,136
TRUE,CTCF_only,down,73
TRUE,CTCF_only,ns,2863
TRUE,CTCF_only,up,180
